# 👑 ALICE DADANG SOVEREIGN — 1-CLICK GEMMA 4 (4B) UNCENSORED TRAINER
**Arsitek & Pencipta**: Commander Dadang Wahyuono (Maestro Dadang)
**Base Model**: `coder3101/gemma-4-E4B-it-heretic` (Google Gemma 4 - 4B Parameter, Full Uncensored & Decensored)
**Dataset**: 36.189 Sampel Doktrin Lelang Storyline v4.2 + CME Order Flow + Office Analytics

### 🚀 Instruksi 1-Klik:
Cukup klik tombol **Play ▶** di bawah SATU KALI. Semua proses berjalan otomatis dari instalasi, download dataset, QLoRA training, sanity check, hingga export GGUF Q4_K_M!

In [ ]:
# ============================================================================== #
# 👑 ALICE DADANG SOVEREIGN — 1-CLICK GEMMA 4 (4B) UNCENSORED MASTER TRAINER
# Arsitek & Pencipta: Commander Dadang Wahyuono (Maestro Dadang)
# Model: coder3101/gemma-4-E4B-it-heretic (Google Gemma 4 - 4B Parameter)
# Fokus: CME Order Flow + Storyline v4.2 + Valid Close Bar + Office Analytics
# ============================================================================== #

import os
import sys
import time
import gc
import shutil
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("=" * 72)
print("👑 ALICE DADANG SOVEREIGN — MEMULAI TRAINING GOOGLE GEMMA 4 (4B)")
print("=" * 72)

# STEP 1: CEK GPU T4
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU tidak terdeteksi! Pastikan di menu Runtime -> Change runtime type sudah pilih T4 GPU!")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ Hardware GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")

# STEP 2: INSTALL DEPENDENSI GEMMA 4
print("\n[1/6] Memeriksa pustaka AI modern...")
os.system("pip install -q -U 'transformers>=4.48.0' 'peft>=0.14.0' 'bitsandbytes>=0.45.0' 'accelerate>=1.2.0' 'datasets>=3.0.0' sentencepiece protobuf gguf")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

# STEP 3: DATASET DOKTRIN (36,189 SAMPEL)
print("\n[2/6] Memeriksa dataset doktrin lelang & office...")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
DATASET_PATH = WORK_DIR / "alice_doctrine_combined.jsonl"

if not DATASET_PATH.exists() or DATASET_PATH.stat().st_size < 1000000:
    print("  Mengunduh 36.189 sampel doktrin langsung dari Miix Tower...")
    os.system(f"curl -L -s -o {DATASET_PATH} https://trade.dadangchatai.com/static/alice_doctrine_combined.jsonl")

sz_mb = DATASET_PATH.stat().st_size / (1024 * 1024)
print(f"✅ Dataset siap: {DATASET_PATH} ({sz_mb:.2f} MB)")

# STEP 4: TOKENIZER GEMMA 4 & TOKENISASI
MODEL_ID = "coder3101/gemma-4-E4B-it-heretic"
print(f"\n[3/6] Memuat Tokenizer & Menyiapkan Model Gemma 4 (4B) ({MODEL_ID})...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 512
raw_ds = load_dataset("json", data_files=str(DATASET_PATH), split="train")

def tokenize_fn(example):
    msgs = example["messages"]
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(full_ids)
    im_end = tokenizer.encode("<turn|>", add_special_tokens=False)
    assist_hdr = tokenizer.encode("<|turn>model\n", add_special_tokens=False)
    if assist_hdr and im_end:
        i, n = 0, len(full_ids)
        h_len, e_len = len(assist_hdr), len(im_end)
        while i <= n - h_len:
            if full_ids[i : i + h_len] == assist_hdr:
                j = i + h_len
                while j <= n - e_len and full_ids[j : j + e_len] != im_end:
                    labels[j] = full_ids[j]
                    j += 1
                if j <= n - e_len:
                    for k in range(e_len):
                        labels[j + k] = full_ids[j + k]
                i = j + e_len
            else:
                i += 1
    if all(x == -100 for x in labels):
        labels = full_ids.copy()
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

tokenized_ds = raw_ds.map(tokenize_fn, remove_columns=raw_ds.column_names, num_proc=2)
tokenized_ds = tokenized_ds.filter(lambda x: len(x["input_ids"]) >= 16 and any(l != -100 for l in x["labels"]))
print(f"✅ Tokenisasi Berhasil: {len(tokenized_ds):,} sampel doktrin siap ditrain pada Gemma 4.")

del raw_ds
gc.collect()
torch.cuda.empty_cache()

# STEP 5: LOAD GEMMA 4 (4B) 4-BIT DENGAN CPU OFFLOAD PERMITTED
print("  Memuat bobot Gemma 4 (4B) ke GPU dengan offload protection...")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    dtype=torch.float16,
    device_map="auto",
    offload_buffers=True,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

for param in base_model.parameters():
    param.requires_grad = False
if hasattr(base_model, "enable_input_require_grads"):
    base_model.enable_input_require_grads()
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

# LoRA Config spesifik untuk Language Model Gemma 4 (Menghindari Gemma4ClippableLinear di vision/audio tower)
peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, peft_cfg)
model.print_trainable_parameters()

# STEP 6: TRAINING GEMMA 4 (BATCH SIZE 1 + ACCUM 16 UNTUK VRAM AMAN)
print("\n[4/6] 🚀 Memulai Training Gemma 4 (4B) Alice Sovereign...")
TMP_DIR = Path("/tmp/alice_build")
TMP_DIR.mkdir(parents=True, exist_ok=True)
LORA_OUT = WORK_DIR / "alice_gemma4_sovereign_lora"

training_args = TrainingArguments(
    output_dir=str(TMP_DIR / "checkpoints"),
    max_steps=250,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    report_to="none",
    seed=3407,
)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=collator,
    processing_class=tokenizer,
)
trainer.train()

model.save_pretrained(str(LORA_OUT))
tokenizer.save_pretrained(str(LORA_OUT))
print(f"✅ Bobot LoRA Gemma 4 tersimpan di: {LORA_OUT}")

# STEP 7: SANITY CHECK INFERENCE
print("\n[5/6] 🧪 Menjalankan Sanity Check Doktrin...")
model.eval()
test_prompts = [
    "Siapa nama lo dan siapa pencipta serta pemilik lo?",
    "Boleh gak kita analisa XAUUSD pakai indikator RSI atau MACD?",
    "Bagaimana hierarki benteng Storyline v4.2 jika M5 menembus arah H1?"
]
for tp in test_prompts:
    txt = tokenizer.apply_chat_template([{"role": "user", "content": tp}], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(txt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=150, do_sample=True, temperature=0.7, top_p=0.9)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n👤 Commander: {tp}\n👑 Alice (Gemma 4): {ans}\n")

# STEP 8: AUTO-EXPORT STANDALONE GGUF Q4_K_M (ZERO-OOM MEMORY GUARD)
print("\n[6/6] 📦 Memulai Otomatisasi Peleburan LoRA & Kuantisasi GGUF Q4_K_M...")
del trainer, model, base_model
gc.collect()
torch.cuda.empty_cache()

# Backup otomatis ke Google Drive jika terpasang
drive_backup = Path("/content/drive/MyDrive/alice_sovereign_backup")
if Path("/content/drive/MyDrive").exists():
    drive_backup.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(LORA_OUT), str(drive_backup / "alice_gemma4_sovereign_lora"), dirs_exist_ok=True)
    print(f"💾 Backup LoRA aman di Google Drive: {drive_backup}")

# Jalankan exporter GGUF anti-OOM resmi dengan 8 GB Swap Shield + In-Place LoRA Merger
os.system("curl -s https://trade.dadangchatai.com/static/export_alice.py | python3")

# Salin GGUF akhir ke Google Drive jika ada
final_gguf = WORK_DIR / "alice-dadang-sovereign-Q4_K_M.gguf"
if final_gguf.exists() and Path("/content/drive/MyDrive").exists():
    shutil.copy2(str(final_gguf), str(drive_backup / final_gguf.name))
    print(f"💾 Master GGUF aman tersalin ke Google Drive: {drive_backup / final_gguf.name}")
